In [15]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [16]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

Q1. How many lesson pages

In [17]:
len(documents)

72

Q2. Indexing and searching

In [18]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [19]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=1
)

search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

Q3. RAG

In [20]:
from dotenv import load_dotenv
load_dotenv()

import os
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client
)

In [21]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

print(f"Input tokens: {assistant.last_response.usage_metadata.prompt_token_count}")

The agentic loop keeps calling the model until it returns a response that contains no function calls.

Specifically, the process works as follows:

1.  The loop is wrapped in a `while` statement (e.g., `while True:`).
2.  Inside the loop, the agent sends the conversation history (including previous tool results) to the LLM.
3.  The code processes the LLM's response. If the response contains a function call, the code executes the tool and appends the result to the conversation history.
4.  A flag (such as `has_function_calls`) is used to track whether any tools were requested in the current iteration.
5.  If no function calls were returned by the model, the loop breaks, ending the agentic process.

As noted in the context, the model decides when to stop. If the model returns a final answer without asking to call any more tools, the agent recognizes that the task is complete and exits the loop.
Input tokens: 7948


Q4. Chunking

In [22]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [23]:
len(chunks)

295

Q5. RAG with chunking

In [24]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

from dotenv import load_dotenv
load_dotenv()

import os
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

print(f"Input tokens: {assistant.last_response.usage_metadata.prompt_token_count}")

The agentic loop uses a `while True` loop to keep calling the model. Inside the loop, it checks the model's response for any function calls:

1. If the model returns **function calls**, the code executes them, appends the results to the message history, and sets a flag (`has_function_calls`) to `True`.
2. The loop continues to the next iteration, sending the updated message history (including the tool results) back to the model.
3. The loop stops when the model returns a response without any function calls, which triggers the `if has_function_calls == False:` condition to break the loop.

In essence, the model itself decides how many times to search, and the loop simply persists until the model provides a final answer rather than requesting further tool use.
Input tokens: 2600


Q6. Turning it into an agent

In [25]:
def search(query: str) -> list:
    """Search the chunk index database for entries matching the given query.

    Args:
        query: The search query text to look up in the document chunks.

    Returns:
        A list of matching document chunks from the minsearch index.
    """
    boost_dict = {"content": 1.0}
    
    results = index.search(
        query=query,
        num_results=5,
        boost_dict=boost_dict
    )
    
    return results

In [26]:
search_tool = {
    "function_declarations": [
        {
            "name": "search",
            "description": "Search the FAQ database for entries matching the given query.",
            "parameters": {
                "type": "OBJECT",
                "properties": {
                    "query": {
                        "type": "STRING",
                        "description": "Search query text to look up in the course FAQ."
                    }
                },
                "required": ["query"]
            }
        }
    ]
}

In [27]:
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""
question = "How does the agentic loop work, and how is it different from plain RAG?"

messages = [
    types.Content(role="user", parts=[types.Part.from_text(text=question)])
]


search_call_counter = 0

def search(query: str) -> list:
    """Search the chunk index database for entries matching the given query."""
    global search_call_counter
    search_call_counter += 1
    
    boost_dict = {"content": 1.0}
    return index.search(
        query=query,
        num_results=5,
        boost_dict=boost_dict
    )

search_call_counter = 0

# agentic loop
messages = [
    types.Content(role="user", parts=[types.Part.from_text(text=question)])
]

while True:
    response = client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=messages,
        config={
            "system_instruction": instructions,
            "tools": [search_tool]
        }
    )
    
    if response.function_calls:
        call = response.function_calls[0]
        print(f"[Agent] Викликаю інструмент '{call.name}' з запитом: '{call.args['query']}'")
        
        results = search(**call.args)
        
        messages.append(response.candidates[0].content)
        messages.append(
            types.Content(
                role="user",
                parts=[types.Part.from_function_response(name=call.name, response={"result": results})]
            )
        )
    else:
        print("\n--- ВІДПОВІДЬ АГЕНТА ---")
        print(response.text)
        break
print(response.text)

[Agent] Викликаю інструмент 'search' з запитом: 'what is the agentic loop'

--- ВІДПОВІДЬ АГЕНТА ---
The fundamental difference between plain RAG and an agentic loop lies in **who is in control of the execution flow.**

### Plain RAG (Fixed, Rigid Flow)
In a plain RAG pipeline, the process is a static, pre-defined sequence of steps that happens regardless of the user's input:
1. **Search:** The system performs a search based on the user's query.
2. **Retrieve:** The results are gathered.
3. **Generate:** The results are passed to the LLM to generate an answer.

Because this flow is "hard-coded," it cannot adapt. If the initial search yields poor results (e.g., due to a typo or ambiguous wording), the pipeline continues blindly, and the LLM receives irrelevant information, often leading to a poor or incorrect final response. There is no mechanism for recovery.

### The Agentic Loop (Adaptive, Intelligent Flow)
An agentic loop puts the LLM in the "driver's seat." Instead of a fixed seque

In [28]:
print(f"Кількість викликів інструменту search: {search_call_counter}")

Кількість викликів інструменту search: 1
